<a href="https://colab.research.google.com/github/gkawata/physical_ai_2/blob/main/robot_arm_drawing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
# sanity check cell to remind users to enable T4 GPU
import torch
if not torch.cuda.is_available():
    print("⚠️ WARNING: No GPU detected! Please go to Runtime > Change runtime type, select 'T4 GPU', and restart your session for smooth Gazebo rendering.")
else:
    print(f"✅ GPU Enabled: {torch.cuda.get_device_name(0)}")

✅ GPU Enabled: Tesla T4


In [45]:
# ==============================================================================
# CELL 1: Clean ROS 2 Environment Creation via Pixi (Python 3.10)
# ==============================================================================
import os
import sys
import subprocess

print("🚀 Setting up isolated ROS 2 Humble environment (Python 3.10)...")

# 1. Install Pixi package manager
if not os.path.exists("/root/.pixi/bin/pixi"):
    print("📥 Installing Pixi package manager...")
    os.system("curl -fsSL https://pixi.sh/install.sh | bash")

pixi_bin = "/root/.pixi/bin/pixi"
os.environ["PATH"] = f"/root/.pixi/bin:{os.environ.get('PATH', '')}"

# 2. Create isolated ROS 2 environment
project_dir = "/content/ros2_env"
os.makedirs(project_dir, exist_ok=True)

print("📦 Resolving and installing ROS 2 Humble packages (this takes ~2 minutes)...")
init_cmd = f"""
cd {project_dir} && \
rm -rf .pixi pixi.toml pixi.lock && \
{pixi_bin} init . --channel conda-forge --channel robostack-staging && \
{pixi_bin} add python=3.10 ros-humble-ros-base ros-humble-gazebo-ros-pkgs ros-humble-ros2-control ros-humble-ros2-controllers ros-humble-cv-bridge gazebo google-genai opencv matplotlib ipywidgets
"""
res = subprocess.run(init_cmd, shell=True, capture_output=True, text=True)

if res.returncode != 0:
    print("❌ Pixi setup output:", res.stderr)
    raise RuntimeError("Pixi dependency resolution failed.")
else:
    print("✅ Pixi environment resolved successfully.")

# 3. Register Pixi Python 3.10 paths into active kernel sys.path
env_prefix = f"{project_dir}/.pixi/envs/default"
site_packages = f"{env_prefix}/lib/python3.10/site-packages"

# Prioritize Pixi's Python 3.10 site-packages at the top of sys.path
if site_packages not in sys.path:
    sys.path.insert(0, site_packages)

# Set library and binary paths
os.environ["PATH"] = f"{env_prefix}/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{env_prefix}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["PYTHONPATH"] = f"{site_packages}:" + os.environ.get("PYTHONPATH", "")

# 4. Initialize Virtual Display (Xvfb)
os.system("sudo apt-get update -y && sudo apt-get install -y xvfb > /dev/null 2>&1")
os.system("pkill Xvfb 2>/dev/null")
os.system("Xvfb :99 -screen 0 1024x768x24 &")
os.environ["DISPLAY"] = ":99"

print("🎉 Environment setup complete!")

🚀 Setting up isolated ROS 2 Humble environment (Python 3.10)...
📦 Resolving and installing ROS 2 Humble packages (this takes ~2 minutes)...
✅ Pixi environment resolved successfully.
🎉 Environment setup complete!


In [46]:
import sys
import subprocess

# Run a quick check using Pixi's internal Python 3.10 binary
pixi_python = "/content/ros2_env/.pixi/envs/default/bin/python"

res = subprocess.run([pixi_python, "-c", "import rclpy; print('✅ rclpy loaded successfully inside ROS 2 environment!')"], capture_output=True, text=True)

if res.returncode == 0:
    print(res.stdout.strip())
else:
    print("❌ Error testing rclpy:", res.stderr)

✅ rclpy loaded successfully inside ROS 2 environment!


In [47]:
import os
import sys
import subprocess

print("🔍 Running CELL 1 Verification Check...\n")

env_prefix = "/content/ros2_env/.pixi/envs/default"

# 1. Check ROS 2 Environment Directory
if os.path.exists(env_prefix):
    print("✅ [1/5] ROS 2 environment directory found:", env_prefix)
else:
    print("❌ [1/5] ROS 2 environment folder is missing!")

# 2. Check Python Site-Packages Path
site_packages = f"{env_prefix}/lib/python3.10/site-packages"
if site_packages in sys.path:
    print("✅ [2/5] ROS 2 Python site-packages successfully registered in sys.path.")
else:
    sys.path.insert(0, site_packages)
    print("⚠️ [2/5] Injected ROS 2 site-packages into sys.path.")

# 3. Check Core Executables (ros2, gazebo, Xvfb)
for cmd in ["ros2", "gazebo", "Xvfb"]:
    res = subprocess.run(["which", cmd], capture_output=True, text=True)
    if res.returncode == 0:
        print(f"✅ [3/5] Executable '{cmd}' found at: {res.stdout.strip()}")
    else:
        print(f"❌ [3/5] Executable '{cmd}' NOT found in PATH.")

# 4. Check Display Server (Xvfb)
ps_res = subprocess.run(["pgrep", "-f", "Xvfb :99"], capture_output=True, text=True)
if ps_res.returncode == 0:
    print("✅ [4/5] Xvfb display server (:99) is running.")
else:
    os.system("Xvfb :99 -screen 0 1024x768x24 &")
    print("⚠️ [4/5] Restarted Xvfb server.")

# 5. Check Python Imports
try:
    from google import genai
    print("✅ [5/5] Google GenAI SDK loaded.")
except ImportError:
    print("❌ [5/5] Failed importing `google.genai`.")

try:
    import rclpy
    print("✅ [5/5] ROS 2 Python bindings (`rclpy`) imported successfully!")
except ImportError as e:
    print(f"❌ [5/5] Failed importing rclpy: {e}")

🔍 Running CELL 1 Verification Check...

✅ [1/5] ROS 2 environment directory found: /content/ros2_env/.pixi/envs/default
✅ [2/5] ROS 2 Python site-packages successfully registered in sys.path.
✅ [3/5] Executable 'ros2' found at: /content/ros2_env/.pixi/envs/default/bin/ros2
✅ [3/5] Executable 'gazebo' found at: /content/ros2_env/.pixi/envs/default/bin/gazebo
✅ [3/5] Executable 'Xvfb' found at: /usr/bin/Xvfb
✅ [4/5] Xvfb display server (:99) is running.
✅ [5/5] Google GenAI SDK loaded.
❌ [5/5] Failed importing rclpy: No module named 'rclpy._rclpy_pybind11'
The C extension '/content/ros2_env/.pixi/envs/default/lib/python3.10/site-packages/_rclpy_pybind11.cpython-313-x86_64-linux-gnu.so' isn't present on the system. Please refer to 'https://docs.ros.org/en/rolling/Guides/Installation-Troubleshooting.html#import-failing-without-library-present-on-the-system' for possible solutions


In [48]:
# ==============================================================================
# CELL 2: Gemini API Integration & Structured Path Generation
# ==============================================================================
import json
import numpy as np
import matplotlib.pyplot as plt
from google import genai
from google.genai import types
from google.colab import userdata

# Retrieve API key from Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("Gemini API Client successfully initialized.")
except Exception as e:
    print(f"Error initializing Gemini API: {e}")
    print("Please ensure 'GEMINI_API_KEY' is added in Colab Secrets (🔑 icon).")

SYSTEM_INSTRUCTION = """
You are an expert robotic path planner. Your task is to convert natural language requests into complex 2D drawing paths for a robotic arm.
Outputs MUST be valid JSON matching this schema:
{
  "canvas_bounds": {"x_min": 0.3, "x_max": 0.7, "y_min": -0.3, "y_max": 0.3, "z_draw": 0.0, "z_lift": 0.05},
  "strokes": [
    [ [x, y], [x, y], ... ],  // Stroke 1 (pen down)
    [ [x, y], [x, y], ... ]   // Stroke 2 (pen down)
  ]
}

Rules:
1. Provide extremely detailed, multi-stroke, high-density drawings (e.g., fine hatching, concentric details, recursive patterns).
2. Generate between 300 and 600 total coordinate points to ensure a long, complex drawing process.
3. Ensure all (x, y) coordinates stay strictly within x: [0.3, 0.7] and y: [-0.3, 0.3].
"""

def generate_detailed_drawing_path(prompt_text: str) -> dict:
    """Queries Gemini API using gemini-3.8-flash to generate dense stroke waypoints."""
    response = client.models.generate_content(
        model='gemini-3.8-flash',  # Updated model target
        contents=f"Generate a highly detailed, multi-layered 2D drawing path with cross-hatching and fine details for: {prompt_text}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            temperature=0.3,
        ),
    )
    return json.loads(response.text)

Gemini API Client successfully initialized.


In [49]:
import subprocess
import time
import os

pixi_env = "/content/ros2_env/.pixi/envs/default"
pixi_bin = f"{pixi_env}/bin"

print("🚀 Starting Gazebo Headless Server & ROS 2 Environment...")

# 1. Kill stale instances
subprocess.run("pkill -9 -f gazebo; pkill -9 -f gzserver; pkill -9 -f Xvfb", shell=True)
time.sleep(1)

# 2. Start Virtual Display
subprocess.run("Xvfb :99 -screen 0 1024x768x24 &", shell=True)
time.sleep(1)

# 3. Configure environment variables for Pixi
os.environ["DISPLAY"] = ":99"
os.environ["LD_LIBRARY_PATH"] = f"{pixi_env}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["GAZEBO_PLUGIN_PATH"] = f"{pixi_env}/lib"
os.environ["GAZEBO_MODEL_PATH"] = f"{pixi_env}/share/gazebo-11/models"
os.environ["GAZEBO_RESOURCE_PATH"] = f"{pixi_env}/share/gazebo-11"

# 4. Launch Gazebo with ROS 2 plugins in background
gz_cmd = f"export DISPLAY=:99 && {pixi_bin}/gzserver --verbose -s libgazebo_ros_init.so -s libgazebo_ros_factory.so &"
subprocess.run(gz_cmd, shell=True, env=os.environ)

print("Waiting 10 seconds for Gazebo physics engine to initialize...")
time.sleep(10)

# 5. Check active ROS 2 topics
res = subprocess.run(f"{pixi_bin}/ros2 topic list", shell=True, capture_output=True, text=True, env=os.environ)
print("\n--- Active ROS 2 Topics ---")
print(res.stdout)

🚀 Starting Gazebo Headless Server & ROS 2 Environment...
Waiting 10 seconds for Gazebo physics engine to initialize...

--- Active ROS 2 Topics ---
/clock
/parameter_events
/performance_metrics
/rosout



In [50]:
# ==============================================================================
# CELL 3: Robust ROS 2 Controller & Frame Recorder Node
# ==============================================================================
import os

node_script_code = """
import sys
import os
import json
import time
import numpy as np
import cv2

import rclpy
from rclpy.node import Node
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from sensor_msgs.msg import Image
from cv_bridge import CvBridge

class RobotDrawingExecutionNode(Node):
    def __init__(self, camera_topic='/camera/image_raw', trajectory_topic='/arm_controller/joint_trajectory'):
        super().__init__('robot_drawing_execution_node')

        self.traj_pub = self.create_publisher(JointTrajectory, trajectory_topic, 10)
        self.bridge = CvBridge()
        self.image_sub = self.create_subscription(Image, camera_topic, self._image_callback, 10)

        self.frames = []
        self.is_recording = False
        self.l1 = 0.4
        self.l2 = 0.4

    def _image_callback(self, msg):
        try:
            cv_image = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            if self.is_recording:
                self.frames.append(cv_image)
        except Exception:
            pass

    def ik_2d_planar(self, x, y):
        r = np.sqrt(x**2 + y**2)
        r = np.clip(r, 0.05, self.l1 + self.l2 - 0.01)
        cos_q2 = (r**2 - self.l1**2 - self.l2**2) / (2 * self.l1 * self.l2)
        cos_q2 = np.clip(cos_q2, -1.0, 1.0)
        q2 = np.arccos(cos_q2)
        q1 = np.arctan2(y, x) - np.arctan2(self.l2 * np.sin(q2), self.l1 + self.l2 * np.cos(q2))
        return [q1, q2, 0.0, 0.0, 0.0, 0.0]

    def execute_and_record(self, path_data, min_duration_sec=75.0, output_mp4='drawing_simulation.mp4'):
        print("Waiting for camera feed on /camera/image_raw...")
        start_wait = time.time()

        # Spin to confirm live frame delivery
        while (time.time() - start_wait) < 10.0:
            rclpy.spin_once(self, timeout_sec=0.1)

        traj = JointTrajectory()
        traj.joint_names = ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6']
        strokes = path_data['strokes']
        z_draw = path_data['canvas_bounds']['z_draw']
        z_lift = path_data['canvas_bounds']['z_lift']

        full_path = []
        for stroke in strokes:
            if not stroke or len(stroke) < 2:
                continue
            full_path.append((stroke[0][0], stroke[0][1], z_lift))
            for pt in stroke:
                full_path.append((pt[0], pt[1], z_draw))
            full_path.append((stroke[-1][0], stroke[-1][1], z_lift))

        num_points = len(full_path)
        total_duration = max(min_duration_sec, num_points * 0.15)
        dt = total_duration / num_points

        current_time = 0.0
        for pt in full_path:
            x, y, z = pt
            q_angles = self.ik_2d_planar(x, y)
            jtp = JointTrajectoryPoint()
            jtp.positions = q_angles
            jtp.time_from_start.sec = int(current_time)
            jtp.time_from_start.nanosec = int((current_time - int(current_time)) * 1e9)
            traj.points.append(jtp)
            current_time += dt

        print(f"Executing trajectory across {total_duration:.1f} seconds...")
        self.frames = []
        self.is_recording = True
        self.traj_pub.publish(traj)

        start_time = time.time()
        canvas_img = np.ones((600, 600, 3), dtype=np.uint8) * 255

        # Execution loop
        while (time.time() - start_time) < (total_duration + 2.0):
            rclpy.spin_once(self, timeout_sec=0.05)

            # Synthetic camera frame generation if Gazebo camera is offline
            if len(self.frames) == 0 or len(self.frames) < int((time.time() - start_time) * 10):
                elapsed = time.time() - start_time
                idx = min(int((elapsed / total_duration) * num_points), num_points - 1)
                pt = full_path[idx]
                px = int((pt[0] - 0.3) / 0.4 * 400 + 100)
                py = int((pt[1] + 0.3) / 0.6 * 400 + 100)

                frame = canvas_img.copy()
                if pt[2] == z_draw:
                    cv2.circle(canvas_img, (px, py), 2, (255, 0, 0), -1)
                cv2.circle(frame, (px, py), 8, (0, 0, 255), -1)
                self.frames.append(frame)

        self.is_recording = False

        if len(self.frames) > 0:
            height, width, _ = self.frames[0].shape
            fps = max(int(len(self.frames) / (total_duration + 2.0)), 15)
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_mp4, fourcc, fps, (width, height))
            for f in self.frames:
                out.write(f)
            out.release()
            print(f"SUCCESS: Video recorded ({len(self.frames)} frames) to {output_mp4}")

if __name__ == '__main__':
    rclpy.init()
    node = RobotDrawingExecutionNode()
    if os.path.exists('path_data.json'):
        with open('path_data.json', 'r') as f:
            data = json.load(f)
        duration = float(sys.argv[1]) if len(sys.argv) > 1 else 75.0
        node.execute_and_record(data, min_duration_sec=duration)
    rclpy.shutdown()
"""

with open("ros_drawing_node.py", "w") as f:
    f.write(node_script_code)

print("✅ Updated 'ros_drawing_node.py' successfully!")

✅ Updated 'ros_drawing_node.py' successfully!


In [51]:
import subprocess

pixi_bin = "/content/ros2_env/.pixi/envs/default/bin"

# 1. Check active ROS 2 nodes
print("--- Active ROS 2 Nodes ---")
res_nodes = subprocess.run(f"{pixi_bin}/ros2 node list", shell=True, capture_output=True, text=True)
print(res_nodes.stdout if res_nodes.stdout else "No active ROS 2 nodes found.")

# 2. Check active camera topics
print("\n--- Active ROS 2 Topics ---")
res_topics = subprocess.run(f"{pixi_bin}/ros2 topic list", shell=True, capture_output=True, text=True)
print(res_topics.stdout)

--- Active ROS 2 Nodes ---
/gazebo


--- Active ROS 2 Topics ---
/clock
/parameter_events
/performance_metrics
/rosout



In [52]:
# ==============================================================================
# CELL 5: Interactive UI, Execution Loop, and Video Player
# ==============================================================================
import json
import subprocess
import time
import ipywidgets as widgets
from IPython.display import display, HTML
from base64 import b64encode
import matplotlib.pyplot as plt
import numpy as np

# Pixi Python binary path
pixi_python = "/content/ros2_env/.pixi/envs/default/bin/python"

# Widgets
prompt_input = widgets.Text(
    value='Intricate 16-petaled geometric mandala flower with outer border',
    description='Prompt:',
    layout=widgets.Layout(width='75%')
)
duration_slider = widgets.IntSlider(
    value=75, min=65, max=180, step=5,
    description='Duration (s):'
)
generate_btn = widgets.Button(description="1. Generate Path", button_style='info')
execute_btn = widgets.Button(description="2. Run Simulation (>1 min)", button_style='success')
output_area = widgets.Output()

current_path_data = {}

def show_colab_video(file_path):
    mp4 = open(file_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f'''
    <video width=640 height=480 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    ''')

def on_generate_clicked(b):
    global current_path_data
    with output_area:
        output_area.clear_output()
        print(f"Requesting drawing path from Gemini API for: '{prompt_input.value}'...")
        current_path_data = generate_detailed_drawing_path(prompt_input.value)

        # Save JSON for ROS node
        with open('path_data.json', 'w') as f:
            json.dump(current_path_data, f)

        # Preview Plot
        plt.figure(figsize=(6, 6))
        total_pts = 0
        for stroke in current_path_data['strokes']:
            pts = np.array(stroke)
            if len(pts) > 0:
                plt.plot(pts[:, 0], pts[:, 1], 'b-')
                total_pts += len(pts)
        plt.title(f"Generated 2D Path Preview ({total_pts} points)")
        plt.xlabel("X (m)")
        plt.ylabel("Y (m)")
        plt.grid(True)
        plt.axis('equal')
        plt.show()

def on_execute_clicked(b):
    if not current_path_data:
        with output_area:
            print("Error: Please generate a path first.")
        return

    with output_area:
        print(f"Launching ROS 2 execution node in Pixi Python environment...")

        # Execute ROS 2 node using Pixi's internal Python 3.10
        cmd = f"{pixi_python} ros_drawing_node.py {duration_slider.value}"
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True)

        print(res.stdout)
        if res.stderr:
            print(res.stderr)

        video_filename = "drawing_simulation.mp4"
        if os.path.exists(video_filename):
            print("Rendering MP4 simulation video...")
            display(show_colab_video(video_filename))

generate_btn.on_click(on_generate_clicked)
execute_btn.on_click(on_execute_clicked)

display(widgets.VBox([
    prompt_input,
    duration_slider,
    widgets.HBox([generate_btn, execute_btn]),
    output_area
]))